In [1]:
!pip -q install GEOparse pandas numpy matplotlib


In [14]:
import GEOparse
import pandas as pd

gds = GEOparse.get_GEO(
    filepath="ignore_dir/GDS3257_full.soft",
    destdir="."
)
expression_df = gds.table

27-Dec-2025 22:21:30 INFO GEOparse - Parsing ignore_dir/GDS3257_full.soft: 
27-Dec-2025 22:21:30 DEBUG GEOparse - DATABASE: Geo
27-Dec-2025 22:21:30 DEBUG GEOparse - DATASET: GDS3257
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_1
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_2
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_3
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_4
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_5
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_6
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_7
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_8
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_9
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_10
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_11
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_12
27-Dec-2025 22:21:30 DEBUG GEOparse - SUBSET: GDS3257_13
27-Dec-2025 22:21:30 DEBUG GEOparse - ANNOTATION: 
27-Dec-2025 22:21:30 ERROR GEOpars

In [4]:
print(expression_df.head())
print(expression_df.columns)


         ID_REF    IDENTIFIER  GSM1684096  GSM1684098  GSM1684100  GSM1684102  \
0  ILMN_1343048  ILMN_1343048         NaN         NaN         NaN         NaN   
1  ILMN_1343049  ILMN_1343049         NaN         NaN         NaN         NaN   
2  ILMN_1343050  ILMN_1343050         NaN         NaN         NaN         NaN   
3  ILMN_1343052  ILMN_1343052         NaN         NaN         NaN         NaN   
4  ILMN_1343059  ILMN_1343059         NaN         NaN         NaN         NaN   

   GSM1684104  GSM1684095  GSM1684097  GSM1684099  ...  Platform_SPOTID  \
0         NaN         NaN         NaN         NaN  ...              NaN   
1         NaN         NaN         NaN         NaN  ...              NaN   
2         NaN         NaN         NaN         NaN  ...              NaN   
3         NaN         NaN         NaN         NaN  ...              NaN   
4         NaN         NaN         NaN         NaN  ...              NaN   

   Chromosome location Chromosome annotation GO:Function GO:Pr

In [15]:
expression_df.to_csv("ignore_dir/GDS3257_full.csv", index=False)


In [ ]:
import matplotlib.pyplot as plt

gene = "TP53"  # change gene name here
gene_row = expression_df[expression_df["IDENTIFIER"] == gene]

if not gene_row.empty:
    plt.plot(gene_row[sample_cols].values[0])
    plt.title(f"Expression of {gene}")
    plt.xlabel("Samples")
    plt.ylabel("Expression level")
    plt.show()


NameError: name 'sample_cols' is not defined

In [8]:
import pandas as pd

# 1. Load the dataset
# Replace 'your_data.csv' with the actual path to your file
df = pd.read_csv('ignore_dir/GDS3257_full.csv', low_memory=False)

# 2. Identify the columns
# Based on your description:
# - ID_REF is the unique feature ID
# - GSM... columns are the samples
# - Gene title, Gene symbol, etc., are annotation metadata

# We set the ID_REF as the index so that it becomes the column headers after rotation
df.set_index('ID_REF', inplace=True)

# 3. Filter for numeric sample columns only (Optional but recommended)
# This separates the expression data from the descriptive text (Gene symbol, etc.)
# sample_cols = [col for col in df.columns if col.startswith('GSM')]
# df_samples = df[sample_cols]

# 4. Rotate (Transpose) the data
# Now rows = Samples (GSM...), columns = Features (ID_REF)
df_transposed = df.transpose()

# 5. Save the file for the next step
df_transposed.to_csv('transposed_GDS3257.csv')

print("Transposition complete. File saved as 'transposed_GDS3257.csv'.")
print(f"New Shape: {df_transposed.shape}") # Should be (Number of Samples, ~20000)

Transposition complete. File saved as 'transposed_GDS3257.csv'.
New Shape: (128, 22283)


In [ ]:
import pandas as pd
from sklearn.feature_selection import VarianceThreshold, SelectKBest, t_test_ind
from sklearn.linear_model import LassoCV

# 1. โหลดข้อมูล (สมมติว่า df มี 22000 columns เป็นยีน และ 107 rows เป็นตัวอย่าง)
# X = ข้อมูลยีน, y = label (Tumor=1, Normal=0)

# 2. Filter: ลบยีนที่มีความแปรปรวนต่ำ (Low Variance)
selector = VarianceThreshold(threshold=0.1) 
X_filtered = selector.fit_transform(X)

# 3. Statistical Selection: เลือกยีนที่เด่นที่สุด 500 อันดับแรกด้วย t-test
# (ในทางปฏิบัติแนะนำให้ใช้ f_classif หรือเขียนฟังก์ชัน t-test เอง)
from sklearn.feature_selection import f_classif
k_best = SelectKBest(score_func=f_classif, k=500)
X_top500 = k_best.fit_transform(X_filtered, y)

# 4. (Optional) ใช้ LASSO เพื่อคัดกรองให้เหลือยีนที่สำคัญจริงๆ
lasso = LassoCV(cv=5).fit(X_top500, y)
important_genes_mask = lasso.coef_ != 0
selected_genes = X_top500.columns[important_genes_mask]